# ESR1 chemogenetic mediation

Analysis source for Supplementary Figure S8: CNO/saline mediation with mouse-indicator covariates and a legacy 60-row smoothing variant. The active filter includes intact-male and female conditions.

See [README](README.md) for inputs and implementation notes. Data and saved outputs are not distributed. The calculation code is retained; headings and inaccurate comments have been clarified.

## Imports and original random seed

In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import sklearn.linear_model as lm 
import re
import statsmodels.api as sm 
import pandas as pd
import h5py

In [ ]:
# set random seed for reproducibility
np.random.seed(42)

## Load ESR1 scores and labels

In [ ]:
data = h5py.File('ESR1_labelsScoresPower.mat','r')

In [ ]:
print(data.keys())

In [ ]:
myDict = {}
for key in data.keys():

    print(key)
    try:
        myDict[key] = data[key].value
    except:
        myDict[key] = data[key]
    # myDict[key] = data[key].value

In [ ]:
labels_list = []
soft_list = []
for key in data.keys():
    if bool(re.search('_key',key)):
        print('key',key)
        ESR1_labels_key = data[key]
    elif bool(re.search('_labels',key)):
        print('labels',key)
        labels_list.append(data[key])
    elif bool(re.search('_soft',key)):
        print('soft',key)
        soft_list.append(data[key])
    else:
        print('!!!!!!!!!!!!!!!!!!!!!!!!')
        print('Unmatched',key)

In [ ]:
ESR1_labels_key.keys()

## Select behavioral conditions and encode outcomes

In [ ]:
# extract labels
behavioral_labels = []
drug_labels = []
condition_labels = []
soft_labels = []
mouseids = []

for i in range(len(labels_list)):
    behavioral_labels.append(np.squeeze(labels_list[i]['behavior'][()]))
    drug_labels.append(np.squeeze(labels_list[i]['Drug'][()]))
    condition_labels.append(np.squeeze(labels_list[i]['condition'][()]))
    soft_labels.append(soft_list[i][()][0])
    mouseids.append(i*np.ones((soft_list[i][()].shape[1])))

# concantenate alls
behavioral_labels = np.concatenate(behavioral_labels)
drug_labels = np.concatenate(drug_labels)
condition_labels = np.concatenate(condition_labels)
soft_labels = np.concatenate(soft_labels)
mouseids = np.concatenate(mouseids)

if True:
    # Conditions are 4 = male, 6 = female, 8 = castrated
    # only keep conditions 4 and 6
    keep_idx = np.where(np.logical_or(condition_labels == 4,condition_labels == 6))[0]
    behavioral_labels = behavioral_labels[keep_idx]
    drug_labels = drug_labels[keep_idx]
    soft_labels = soft_labels[keep_idx]
    mouseids = mouseids[keep_idx]








In [ ]:
treatment=drug_labels
outcome=behavioral_labels
outcome[outcome==0]=0 # keep non-interaction
outcome[outcome==2]=0 # code non-aggressive interaction into the same as non-interaction
outcome[outcome==3]=np.NaN # remove missing data
M=soft_labels
# convert mouseids to one-hot
mouseids_onehot=np.zeros((len(mouseids),len(np.unique(mouseids))))
for i in range(len(mouseids)):
    mouseids_onehot[i,int(mouseids[i])]=1



In [ ]:
# for each animal, get the number of positive outcomes under each treatment
# and the number of trials under each treatment
for mouse in np.unique(mouseids):
    print('mouse',mouse)
    print('positive outcomes',np.sum(outcome[mouseids==mouse]))
    print('positive outcomes under treatment',np.sum(outcome[(mouseids==mouse) & (drug_labels==1)]))
    print('positive outcomes under control',np.sum(outcome[(mouseids==mouse) & (drug_labels==0)]))
    print('total trials',len(outcome[mouseids==mouse]))
    print('fraction positive',np.sum(outcome[mouseids==mouse])/len(outcome[mouseids==mouse]))
    print('ratio under treatment',np.sum(outcome[(mouseids==mouse) & (drug_labels==1)])/np.sum(outcome[(mouseids==mouse) & (drug_labels==0)]))

## Exploratory unadjusted analyses

In [ ]:
## Prepare data for an exploratory mixed-effects fit and unadjusted mediation.
import pandas as pd
import statsmodels.genmod.families.links as links
df = pd.DataFrame({'treatment':treatment,'M':M,'outcome':outcome})
# add mouseids
df.insert(0,'mouseids',mouseids)
# drop nas
df=df.dropna()
df['const']=1

### Separate mixed-effects fit

This diagnostic fit is not passed to the subsequent mediation calculation.

In [ ]:
# create a linear mixed effect model
import statsmodels.api as sm
import statsmodels.formula.api as smf
mixed_mediator_model=smf.mixedlm("M ~ treatment",df,groups=df['mouseids'])
mixed_mediator_results=mixed_mediator_model.fit()
print(mixed_mediator_results.summary())


### Unadjusted mediation

In [ ]:
# Unadjusted mediation: OLS mediator and binomial GLM outcome with a probit link.
probit = links.probit
mediator_model =  sm.OLS.from_formula("M ~ treatment",df)   
outcome_model = sm.GLM.from_formula("outcome ~ M + treatment",
                                    df, family=sm.families.Binomial(link=probit()))
causal_model= sm.stats.Mediation(outcome_model,mediator_model, exposure="treatment",mediator="M").fit()
# pull out ACME and ADE
acme = np.mean(causal_model.ACME_avg)
ade = np.mean(causal_model.ADE_avg)

In [ ]:
causal_model.summary()

## Mediation adjusted for mouse indicators — Supplementary Figure S8

In [ ]:
## causal mediation analysis
import pandas as pd
import statsmodels.genmod.families.links as links
df = pd.DataFrame({'treatment':treatment,'M':M,'outcome':outcome})
# add mouseids
for i in range(len(np.unique(mouseids))):
    df['mouseid'+str(i)]=mouseids_onehot[:,i]
# drop nas
df=df.dropna()

In [ ]:
mediator_model = sm.OLS.from_formula("M ~ treatment + mouseid1 + mouseid2 + mouseid3 + mouseid4 + mouseid5 + mouseid6 + mouseid7", df)
mediator_result = mediator_model.fit()
print(mediator_result.summary())

In [ ]:
probit = links.probit
outcome_model = sm.GLM.from_formula("outcome ~ M + treatment + mouseid1 + mouseid2 + mouseid3 + mouseid4 + mouseid5 + mouseid6 + mouseid7",
                                    df, family=sm.families.Binomial(link=probit()))
causal_model= sm.stats.Mediation(outcome_model, mediator_model, exposure="treatment",mediator="M").fit()
# pull out ACME and ADE
acme = np.mean(causal_model.ACME_avg)
ade = np.mean(causal_model.ADE_avg)

In [ ]:
causal_model.summary()

In [ ]:
df[:5]

## Legacy 60-row smoothing variant — Supplementary Figure S8

The retained code uses a centered sum across concatenated filtered observations. It does not implement a trailing, per-session average. This calculation is preserved rather than silently changed.

In [ ]:
# copy the df into a new matrix
tmp=df.copy().to_numpy()
# Legacy smoothing: centered 60-row sum over the concatenated, filtered observations.
# This is not a trailing average and is not grouped by mouse or session.
# define a convolution kernel
kernel=np.ones(60)
# convolve the mediator
mediator_smoothed=np.convolve(tmp[:,1],kernel,'same')
# redefine kernel to not actually smooth
kernel=np.ones(1)
# convolve the outcome
treatment_smoothed=np.convolve(tmp[:,0],kernel,'same')
# convolve the outcome
outcome_smoothed=np.round(np.convolve(tmp[:,2],kernel,'same'))
# convolve the mouseids (1d over time)
mouseids_smoothed=np.zeros((len(outcome_smoothed),7))
for i in range(7):
    mouseids_smoothed[:,i]=np.convolve(tmp[:,3+i],kernel,'same')
# create a df2
df2=pd.DataFrame({'treatment':treatment_smoothed,'M':mediator_smoothed,'outcome':outcome_smoothed})
# add mouseids
for i in range(7):
    df2['mouseid'+str(i+1)]=mouseids_smoothed[:,i]


In [ ]:
mediator_model = sm.OLS.from_formula("M ~ treatment + mouseid1 + mouseid2 + mouseid3 + mouseid4 + mouseid5 + mouseid6 + mouseid7", df2)
mediator_result = mediator_model.fit()
print(mediator_result.summary())

In [ ]:
probit = links.probit
outcome_model = sm.GLM.from_formula("outcome ~ M + treatment + mouseid1 + mouseid2 + mouseid3 + mouseid4 + mouseid5 + mouseid6 + mouseid7",
                                    df2, family=sm.families.Binomial(link=probit()))
causal_model= sm.stats.Mediation(outcome_model, mediator_model, exposure="treatment",mediator="M").fit()
# pull out ACME and ADE
acme = np.mean(causal_model.ACME_avg)
ade = np.mean(causal_model.ADE_avg)

In [ ]:
print(causal_model.summary())